In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split
import re
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from collections import defaultdict
import numpy as np
from imblearn.over_sampling import RandomOverSampler
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler



In [ ]:
df_train= pd.read_csv('Path/to/saved/df_train_binary_resampled.csv') # <-- Update the path to your resampled training dataset
df_val= pd.read_csv('Path/to/saved/df_val_binary.csv') # <-- Update the path to your validation dataset
df_test= pd.read_csv('Path/to/saved/df_test_binary.csv') # <-- Update the path to your test dataset

df_test

In [ ]:
df_train = df_train.drop_duplicates()

In [ ]:
df_train.rename(columns = {'labels': 'label'}, inplace = True)

In [ ]:
df_train = pd.concat([df_train, df_val], axis = 0)

In [ ]:
df_train.drop(columns = ['subject_id'], inplace = True)
df_train["subject_id"] = df_train["admission_id"].str.split("_").str[0]


In [ ]:
def comma_tokenizer(text):
    return [token.strip() for token in text.split(',')]


TEXT_COL = "input_ids"
LABEL_COL = "label"
GROUP_COL = "subject_id"

text_pipeline = CountVectorizer(
    tokenizer=comma_tokenizer,
    token_pattern=None,
    
)

preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_pipeline, TEXT_COL),
    ],
    sparse_threshold=0.3
)


In [ ]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
#from xgboost import XGBClassifier
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    average_precision_score,
    confusion_matrix
)
import numpy as np

def compute_metrics(y_true, y_pred, y_prob=None):
    metrics = {}

    # ---- Accuracy
    metrics["accuracy"] = accuracy_score(y_true, y_pred)

    # ---- Precision / Recall / F1 per class
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0
    )

    metrics["precision_0"] = precision[0]
    metrics["recall_0"] = recall[0]
    metrics["f1_0"] = f1[0]

    metrics["precision_1"] = precision[1]
    metrics["recall_1"] = recall[1]
    metrics["f1_1"] = f1[1]

    if y_prob is not None:
        if y_prob.ndim == 2 and y_prob.shape[1] >= 2:
            # probability of class 0 (clinically positive)
            pos_probs = y_prob[:, 0]
        else:
            pos_probs = y_prob.ravel()

        y_true_pos = (y_true == 0).astype(int)

        metrics["prauc"] = average_precision_score(y_true_pos, pos_probs)
    else:
        metrics["prauc"] = np.nan

    return metrics


In [ ]:
from sklearn.model_selection import GroupKFold, GridSearchCV, StratifiedKFold, StratifiedGroupKFold
from sklearn.base import clone


pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("oversample", RandomOverSampler(random_state=42)),
        ("clf", LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=42
        ))
    ]
)

models = {

    "Random Forest": (
        Pipeline([
            ("preprocess", preprocessor),
            ("oversample", RandomOverSampler(random_state=42)),
            ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
        ]),
        {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [None, 20, 40],
            "clf__min_samples_leaf": [1, 5],
            'clf__min_samples_split': [2, 5, 10]
        }
    ),

    "GradientBoosting": (
        Pipeline([
            ("preprocess", preprocessor),
            ("oversample", RandomOverSampler(random_state=42)),
            ("clf", GradientBoostingClassifier(random_state=42))
        ]),
        {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [3, 5],
            "clf__learning_rate": [0.05, 0.1]
        }
    ),
    "DecisionTree": (
        Pipeline([
            ("preprocess", preprocessor),
            ("oversample", RandomOverSampler(random_state=42)),
            ("clf", DecisionTreeClassifier(random_state=42))
        ]),
        {
            "clf__max_depth": [None, 5, 10, 20],
            "clf__min_samples_split": [2, 5, 10]
        }
    )
}

outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

nested_results = []

X = df_train[[TEXT_COL]]
y = df_train[LABEL_COL]
groups = df_train[GROUP_COL]

for model_name, (pipe, param_grid) in models.items():

    print(f"\nNested CV for {model_name}")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring="f1",
            cv=inner_cv,
            n_jobs=-1
        )

        gs.fit(X_tr, y_tr, groups=groups.iloc[train_idx])
        best_model = gs.best_estimator_
        best_params = gs.best_params_

        y_prob = best_model.predict_proba(X_te)[:,1]
        y_pred = best_model.predict(X_te)
        y_prob = (
            best_model.predict_proba(X_te)
            if hasattr(best_model, "predict_proba")
            else None
        )

        metrics = compute_metrics(y_te, y_pred, y_prob)
        metrics["model"] = model_name
        metrics["outer_fold"] = fold_idx
        metrics["best_params"] = best_params

        nested_results.append(metrics)

nested_df = pd.DataFrame(nested_results)

params_df = nested_df.loc[:, ["model", "best_params"]].copy()
nested_df.drop(columns = ['best_params'], inplace=True)
summary = (
    nested_df
    .groupby("model")
    .agg(["mean", "std"])
)

summary


In [ ]:
rf_params_df = params_df[params_df["model"] == "Random Forest"]

best_rf_params = (
    rf_params_df["best_params"]
    .value_counts()
    .idxmax()
)

In [ ]:
dt_params_df = params_df[params_df["model"] == "DecisionTree"]

best_dt_params = (
    dt_params_df["best_params"]
    .value_counts()
    .idxmax()
)

In [ ]:
gb_params_df = params_df[params_df["model"] == "GradientBoosting"]

best_gb_params = (
    gb_params_df["best_params"]
    .value_counts()
    .idxmax()
)

In [ ]:
X_train = df_train[[TEXT_COL]]
y_train = df_train[LABEL_COL]

X_test = df_test[[TEXT_COL]]
y_test = df_test[LABEL_COL]

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, recall_score
import joblib
best_params_by_model = {
    "Random Forest": best_rf_params,
    "GradientBoosting": best_gb_params, 
    "DecisionTree": best_dt_params
}

test_results = []

for name, params in best_params_by_model.items():
    pipe = clone(models[name][0])
    pipe.set_params(**params)
    pipe.fit(X_train, y_train)

    if name == "Random Forest":
        joblib.dump(pipe, "Path/to/save/best/model.joblib") # <-- Update the path to save your best model

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None

    row = {
        "model": name,
        "params": params,
        "acc": accuracy_score(y_test, y_pred), 
        "bacc": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_prob) if y_prob is not None else None,
        "sensitivity": recall_score(y_test, y_pred, pos_label = 1), 
        "specificity": recall_score(y_test, y_pred, pos_label = 0)
    }
    test_results.append(row)
    

In [ ]:
results_df = pd.DataFrame(test_results)

In [ ]:
results_df

In [ ]:
results_df.to_csv('Path/to/save/results.csv') # <-- Update the path to save your results CSV

In [ ]:
summary.to_csv('Path/to/save/summary.csv') # <-- Update the path to save your summary CSV

In [ ]:
final_pipe = Pipeline([
    ("preprocess", preprocessor),
    ("oversample", RandomOverSampler(random_state=42)),
    ("clf", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

# set pipeline-style params
final_pipe.set_params(**best_rf_params)

final_pipe.fit(X, y)

In [ ]:
rf = final_pipe.named_steps["clf"]
importances = rf.feature_importances_


def get_feature_names(preprocessor):
    feature_names = []

    for name, transformer, cols in preprocessor.transformers_:
        if transformer == "drop":
            continue

        if hasattr(transformer, "get_feature_names_out"):
            names = transformer.get_feature_names_out(cols)
        else:
            names = cols

        feature_names.extend(names)

    return np.array(feature_names)

feature_names = get_feature_names(
    final_pipe.named_steps["preprocess"]
)

In [ ]:
X_vec = final_pipe.named_steps['preprocess'].transform(X_train)

In [ ]:
#feature_names = vectorizer.get_feature_names_out()
importances = rf.feature_importances_

X_array = X_vec.toarray()
freq_class_1 = X_array[y_train == 1].mean(axis=0)
freq_class_0 = X_array[y_train == 0].mean(axis=0)

direction = np.sign(freq_class_1 - freq_class_0)

# Multiply importance by direction to simulate signed importance
signed_importance = importances * direction

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
    'signed_importance': signed_importance
})


In [ ]:
top_positive = importance_df.sort_values(by='signed_importance', ascending=False).head(20)
top_negative = importance_df.sort_values(by='signed_importance').head(20)

plt.figure(figsize=(16, 10))

# Positive contributions
plt.subplot(2, 1, 1)
plt.barh(top_positive['feature'][::-1], top_positive['signed_importance'][::-1], color='green', height = 0.5)
plt.title('Top 20 Features Increasing Long Stay Probability')
plt.xlabel('Signed Importance')
plt.tight_layout()

# Negative contributions
plt.subplot(2, 1, 2)
plt.barh(top_negative['feature'][::-1], top_negative['signed_importance'][::-1], color='red', height = 0.5)
plt.title('Top 20 Features Decreasing Long Stay Probability')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.savefig("Path/to/save/figure.png", dpi=300, bbox_inches='tight') # <-- Update the path to save your figure
plt.show()


# bias analysis

In [ ]:
df_test['sex_name'] = df_test['input_ids'].str.extract(r'sex:\s*([^,]+)')
df_test['ethnicity'] = df_test['input_ids'].str.extract(r'ethnicity:\s*([^,]+)')

In [ ]:
ethnicity_mapping = {
    # White
    'White British': 'White',
    'Other White Background': 'White',
    'White': 'White',
    'Irish': 'White',
    'White-E. European': 'White',
    'White-Russian': 'White',
    'White-Brazilian': 'White',
    'White-Portuguese': 'White',
    
    # Black
    'Black African': 'Black',
    'Black Caribbean': 'Black',
    'Other Black Background': 'Black',
    
    # Asian
    'Asian Pakistani': 'Asian',
    'Asian Indian': 'Asian',
    'Other Asian Background': 'Asian',
    'Asian Bangladeshi': 'Asian',
    'Other Chinese': 'Asian',
    'Asian-Chinese': 'Asian',
    'Asian-South East Asian': 'Asian',
    'Asian-Indian': 'Asian',
    'Asian-Korean': 'Asian',
    
    # Mixed
    'Mixed White and Asian': 'Mixed',
    'Mixed White and Black Caribbean': 'Mixed',
    'Mixed White and Black African': 'Mixed',
    'Other Mixed Background': 'Mixed',
    
    # Other
    'Other Ethnic Backgrounds': 'Other',
    'Prefer Not To Say': 'Other',
    'N/A': 'Other',
    
    # Not Available
    'Unknown': 'Not Available',
    'Unable to Obtain': 'Not Available',
    'Patient Declined': 'Not Available'
}

df_test['broad_ethnicity'] = df_test['ethnicity'].map(ethnicity_mapping)

In [ ]:
y_pred = final_pipe.predict(X_test)

In [ ]:
results = df_test.copy()
results['prediction'] = y_pred

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = results['label']
y_pred = results['prediction']

confusion_matrix(y_true, y_pred)

In [ ]:
sex_metrics = []

for sex, group in results.groupby('sex_name'):
    y_true = group['label']
    y_pred = group['prediction']
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0,1])
    acc=accuracy_score(y_true, y_pred)

    for cls in [0, 1]:
        sex_metrics.append({
                'Sex': sex,
                'Class': f'Class {cls}',
                'Precision': p[cls],
                'Recall': r[cls],
                'F1': f1[cls],
                'Accuracy': acc
            })
sex_metrics = pd.DataFrame(sex_metrics)
sex_metrics
            

In [ ]:
ethnicity_metrics = []

for ethnicity, group in results.groupby('broad_ethnicity'):
    y_true = group['label']
    y_pred = group['prediction']
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0,1])
    acc=accuracy_score(y_true, y_pred)

    for cls in [0, 1]:
        ethnicity_metrics.append({
                'Ethnicity': ethnicity,
                'Class': f'Class {cls}',
                'Precision': p[cls],
                'Recall': r[cls],
                'F1': f1[cls],
                'Accuracy': acc
            })
ethnicity_metrics = pd.DataFrame(ethnicity_metrics)
ethnicity_metrics
            

In [ ]:
n_boot = 100

y_pred = final_pipe.predict(X_test)
results = df_test.copy()
results['prediction'] = y_pred

sex_metrics = []

eth_metrics = []

for _ in range(n_boot):
    sample = results.sample(len(results), replace = True)
    for sex, g in sample.groupby("sex_name"):
        p,r,f1,_ = precision_recall_fscore_support(
            g["label"], g['prediction'], labels = [0, 1], zero_division=0)
        acc = accuracy_score(g['label'], g['prediction'])

        for cls in [0,1]:
            sex_metrics.append({
                    'Sex': sex,
                    'Class': cls,
                    'Precision': p[cls],
                    'Recall': r[cls],
                    'F1': f1[cls],
                    'Accuracy': acc
                })

    for eth, g in sample.groupby("broad_ethnicity"):
        p,r,f1,_ = precision_recall_fscore_support(
        g["label"], g['prediction'], labels = [0, 1], zero_division=0)
        acc = accuracy_score(g['label'], g['prediction'])
        for cls in [0, 1]:
            eth_metrics.append({
                    'Ethnicity': eth,
                    'Class': cls,
                    'Precision': p[cls],
                    'Recall': r[cls],
                    'F1': f1[cls],
                    'Accuracy': acc
                })
            

In [ ]:
sex_metrics = pd.DataFrame(sex_metrics)
eth_metrics = pd.DataFrame(eth_metrics)


In [ ]:
sex_summary = sex_metrics.groupby(['Sex', 'Class']).agg(['mean', 'std'])
eth_summary = eth_metrics.groupby(['Ethnicity', 'Class']).agg(['mean', 'std'])

In [ ]:
sex_summary

In [ ]:
eth_summary

In [ ]:
df_test.groupby(['broad_ethnicity', 'label']).size().unstack(fill_value=0)

# Traffic light

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

probs_pos = final_pipe.predict_proba(X_test)[:,1]
y_true = y_test.values

# find threshold that maximises Youden's J (sens + spec - 1) 
fpr, tpr, thr = roc_curve(y_true, probs_pos)   
specificity = 1 - fpr
youden_j = tpr + specificity - 1
best_ix = np.argmax(youden_j)
best_thr = thr[best_ix]
best_sens = tpr[best_ix]
best_spec = specificity[best_ix]

print(f"Optimal threshold (Youden's J): {best_thr:.4f}")
print(f"Sensitivity @ thr*: {best_sens:.3f}")
print(f"Specificity @ thr*: {best_spec:.3f}")

In [ ]:
y_pred = (probs_pos >= best_thr).astype(int)

cm = confusion_matrix(y_true, y_pred, labels=[0,1])
tn, fp, fn, tp = cm.ravel()
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
sens = recall_score(y_true, y_pred)                 # recall = sensitivity
f1 = f1_score(y_true, y_pred)
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("Confusion matrix [rows=True class, cols=Pred]:")
print(cm)
print(f"Accuracy: {acc:.3f}  Precision: {prec:.3f}  Sensitivity: {sens:.3f}  Specificity: {spec:.3f},  F1: {f1:.3f}")
